# Deep Learning Homework n. 1

We implement forward and backward passes for a fully connected network using
PyTorch tensors. The derivatives are calculated manually, without autograd.


## Imports


In [3]:
from typing import Protocol
import torch
from dataclasses import dataclass

Tensor = torch.Tensor  # avoid typing `torch.Tensor` at every function declaration

## Setting up the network


### Activation functions

We define an `Activation` protocol so that each layer can use any of the
activation functions below. Each activation provides `f` for the function and
`df` for its element-wise derivative at $z$. For ReLU at $z=0$, we use 0.


In [4]:
class Activation(Protocol):
    def f(self, z: Tensor) -> Tensor: ...
    def df(self, z: Tensor) -> Tensor: ...
    def __repr__(self) -> str:
        return type(self).__name__


class ReLU(Activation):
    def f(self, z: Tensor) -> Tensor:
        return z.clamp(min=0)

    def df(self, z: Tensor) -> Tensor:
        return (z > 0).to(z.dtype)


class Identity(Activation):
    def f(self, z: Tensor) -> Tensor:
        return z

    def df(self, z: Tensor) -> Tensor:
        return torch.ones_like(z)


class Sigmoid(Activation):
    def f(self, z: Tensor) -> Tensor:
        return 1 / (1 + torch.exp(-z))

    def df(self, z: Tensor) -> Tensor:
        s = self.f(z)
        return s * (1 - s)


### Loss functions

Similarly, we define a `Loss` protocol. Here, `df` returns the derivative with
respect to the prediction $\hat y$.

For RSS, we use $\mathcal L=\sum_i(\hat y_i-y_i)^2$, whose derivative with respect
to the prediction is $2(\hat y-y)$.
For absolute error, the derivative is $\operatorname{sign}(\hat y-y)$, with 0
when the prediction equals the target.

For binary log-loss, the targets are $y_i\in\{0,1\}$ and the output layer uses
Sigmoid. The loss is

$$
-\sum_i[y_i\log(p_i)+(1-y_i)\log(1-p_i)]
$$

We clip the predictions away from 0 and 1 to avoid $\log(0)$.


In [5]:
class Loss(Protocol):
    def verify_dim_match(self, y_hat: Tensor, y: Tensor) -> None:
        if y_hat.shape != y.shape:
            raise ValueError(
                f"Expected matching shapes, got {y_hat.shape} and {y.shape}. Go check your dimensions ;)"
            )

    def f(self, y_hat: Tensor, y: Tensor) -> Tensor: ...
    def df(self, y_hat: Tensor, y: Tensor) -> Tensor: ...


class RSS(Loss):
    def f(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return ((y_hat - y) ** 2).sum()

    def df(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return 2 * (y_hat - y)


class LogLoss(Loss):  # binary targets; use a Sigmoid output layer
    def _clamp(self, y_hat: Tensor) -> Tensor:
        return y_hat.clamp(1e-7, 1 - 1e-7)

    def f(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        p = self._clamp(y_hat)
        return -1 * (y * torch.log(p) + (1 - y) * torch.log(1 - p)).sum()

    def df(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        p = self._clamp(y_hat)
        return torch.where(y_hat == p, -y / p + (1 - y) / (1 - p), 0.0)


class AbsoluteError(Loss):
    def f(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return torch.abs(y_hat - y).sum()

    def df(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return (y_hat - y).sign()


### Helper objects

We use dataclasses to group related values and access them by name.
`ForwardCache` stores $h^{(l-1)}$ and $z^{(l)}$ for the backward pass, while
`GradCache` stores $db$ and $dW$. `RunResult` contains the results of a run,
including both the error $\hat y-y$ and the scalar loss.


In [6]:
@dataclass
class LayerSpec:
    size: int
    activation: Activation


@dataclass
class ForwardCache:
    h_prev: Tensor
    z: Tensor


@dataclass
class GradCache:
    db: Tensor
    dW: Tensor


@dataclass
class RunResult:
    x: Tensor
    y: Tensor
    y_hat: Tensor
    err: Tensor
    loss: Tensor
    grads: list[GradCache]


## Implementing the network

We split the implementation into `Layer`, which handles a single layer, and
`Network`, which stores the layers in order and runs the forward and backward passes.


### Layer


Each layer receives its input and output sizes, an activation, and a seeded random
generator. We initialize $W\in\mathbb{R}^{out\times in}$ with independent
$\mathcal{N}(0,1)$ entries and $b\in\mathbb{R}^{out}$ with zeros.

#### Forward pass

The layer receives the previous layer's output $h^{(l-1)}$ and calculates
$z^{(l)}=Wh^{(l-1)}+b$, then returns $h^{(l)}=f(z^{(l)})$.
We store $h^{(l-1)}$ and $z^{(l)}$ in the cache for the backward pass.

#### Backward pass

We first multiply the incoming gradient by the activation's derivative:
$\delta=\dfrac{\partial\mathcal L}{\partial h^{(l)}}\odot f'(z^{(l)})$.
Then $dW=\delta(h^{(l-1)})^\top$ and $db=\delta$.
The gradient passed to the previous layer is $W^\top\delta$.
We store $dW$ and $db$ so they can be returned at the end of the run.

#### Batch (bonus)

We also added batch support as a bonus. Each row of the input matrix represents
one example, so the forward calculation becomes $Z=H_{\mathrm{prev}}W^\top+b$,
with the bias added to every row.

In the backward pass, $\Delta$ contains one $\delta$ per row. We calculate
$dW=\Delta^\top H_{\mathrm{prev}}$, sum the rows of $\Delta$ to get $db$,
and pass $\Delta W$ to the previous layer.


In [7]:
STEP_SIZE = 0.03


class Layer:
    def __init__(self, specs: LayerSpec, in_size: int, g: torch.Generator):
        self.activation = specs.activation
        self.W = torch.randn(specs.size, in_size, generator=g)
        self.b = torch.zeros(specs.size)

        self.cache: ForwardCache | None = None
        self.grad: GradCache | None = None

    def forward(self, h_prev) -> Tensor:
        z = (
            self.W @ h_prev + self.b
            if h_prev.dim() == 1  # col vec
            else h_prev @ self.W.T + self.b  # rows are the vecs
        )
        self.cache = ForwardCache(h_prev, z)
        return self.activation.f(z)

    def backward(self, dh, step=False) -> Tensor:
        if not self.cache:
            raise RuntimeError("Layer has no forward cache")
        c = self.cache

        z, h_prev = c.z, c.h_prev
        delta = dh * self.activation.df(z)

        if h_prev.dim() == 1:  # vector (col vec)
            dW = torch.outer(delta, h_prev)
            db = delta
            prev_dh = self.W.T @ delta
        else:
            dW = delta.T @ h_prev  # batch (the rows are the vecs)
            db = delta.sum(0)
            prev_dh = delta @ self.W

        self.grad = GradCache(db, dW)

        if step:
            self.W = self.W - STEP_SIZE * dW
            self.b = self.b - STEP_SIZE * db
        return prev_dh

    def get_grad(self) -> GradCache:
        if self.grad is None:
            raise RuntimeError("Layer has no gradients; run backward first")
        return self.grad

    def clear(self):
        self.cache = None
        self.grad = None


### The network

`layers_specs` gives the size and activation of each layer, including the output
layer. Each layer's output size becomes the next layer's input size. We use a
single generator with seed 42 to make the initialization reproducible.
The layers keep their weights and biases between calls to `run`.

In `run`, we pass $x$ through the layers, calculate the loss derivative, and
traverse the layers in reverse order for backpropagation. The gradients are
returned in the original layer order.

#### Training and epochs (bonus)

As another bonus, we added optional training through the `epocs` argument to
`run`. Its default value is 0, so a standard run calculates the outputs and
gradients without updating the weights.

For a positive `epocs` value, each epoch performs a forward and backward pass
on the supplied input, followed by a gradient descent update of the weights and
biases with `STEP_SIZE = 0.03`. After the final update, we perform one more
forward and backward pass so that the returned outputs and gradients correspond
to the updated network.


In [8]:
class Network:
    def __init__(self, in_size: int, loss: Loss, layers_specs: list[LayerSpec]):
        self.in_size = in_size
        self.depth = len(layers_specs)
        self.loss = loss
        self.layers = self.make_layers(layers_specs)

    def make_layers(self, layers_specs: list[LayerSpec]) -> list[Layer]:
        g = torch.Generator().manual_seed(42)
        layers = []
        in_size = self.in_size
        for spec in layers_specs:
            layers.append(Layer(spec, in_size, g))
            in_size = spec.size
        return layers

    def run(self, x: Tensor, y: Tensor, epocs: int = 0) -> RunResult:
        if epocs < 0:
            raise ValueError("epocs must be at least 0")
        dtype = torch.get_default_dtype()
        x = torch.as_tensor(x, dtype=dtype)
        y = torch.as_tensor(y, dtype=dtype)

        y_hat = x
        for i in range(epocs + 1):  # optional updates, followed by one evaluation
            y_hat = x
            for l in self.layers:
                y_hat = l.forward(y_hat)
            dh = self.loss.df(y_hat, y)
            for l in reversed(self.layers):
                dh = l.backward(dh, step=i < epocs)

        return RunResult(
            x,
            y,
            y_hat,
            y_hat - y,
            self.loss.f(y_hat, y),
            [l.get_grad() for l in self.layers],
        )


## Displaying the results


### LaTeX display helpers

We use LaTeX formatting to display the matrices and derivatives in the notebook.


In [9]:
CAN_TEXIFY = False
try:
    from IPython import get_ipython
    from IPython.display import display, Latex

    CAN_TEXIFY = get_ipython() is not None
except ImportError:
    pass


def fmt(x) -> str:
    return f"{x:.4f}".rstrip("0").rstrip(".")


def texify(t: Tensor) -> str:
    if not CAN_TEXIFY:
        return t.__repr__()
    if t.dim() == 0:
        return fmt(t.item())
    if t.dim() == 1:
        rows = [fmt(x) for x in t]  # column vector
    else:
        rows = [" & ".join(fmt(x) for x in row) for row in t]

    body = (r" \\" + "\n").join(rows)
    return r"\begin{bmatrix}" + "\n" + body + "\n" + r"\end{bmatrix}"


def partial(symb: str, l: int) -> str:
    return rf"\frac{{\partial \mathcal{{L}}}}{{\partial {symb}^{{({l})}}}}"


def show(tex: str, plain: str) -> None:
    if CAN_TEXIFY:
        display(Latex(tex))
    else:
        print(plain)


### Displaying the network's weights

This function expands the forward expression using the current weights and
biases, allowing us to inspect the network as a single expression.


In [ ]:
def show_network(net: Network) -> None:
    sizes = [l.W.shape[0] for l in net.layers]

    def net_layer(i: int) -> str:
        if i < 0:
            return "x"
        l = net.layers[i]
        h_prev = net_layer(i - 1)
        body = rf"{texify(l.W)}{h_prev} + {texify(l.b)}"
        if isinstance(l.activation, Identity):
            return body
        return rf"{l.activation!r}\left({body}\right)"

    show(
        rf"$$ \hat{{y}} = {net_layer(net.depth - 1)} $$",
        f"Network: input {net.in_size}, hidden {sizes[:-1]}, output {sizes[-1]}",
    )


### Displaying a run report

The report shows the input, target, prediction, error, loss, and gradients for
each layer. We number the layers from 1, as in the equations above.


In [11]:
def report(r: RunResult) -> None:
    show(rf"$x = {texify(r.x)},\; y = {texify(r.y)}$", f"x = {r.x}\ny = {r.y}")
    show(
        rf"$\hat{{y}} = {texify(r.y_hat)},\; \text{{Error}} = {texify(r.err)},\; "
        rf"\mathcal{{L}} = {fmt(r.loss.item())}$",
        f"y_hat = {r.y_hat}\nerror = {r.err}\nloss = {r.loss.item():.4f}",
    )
    for num, g in enumerate(r.grads, start=1):
        show(
            rf"$\displaystyle {partial('W', num)} = {texify(g.dW)},\; "
            rf"{partial('b', num)} = {texify(g.db)}$",
            f"dW^({num}) = {g.dW}\ndb^({num}) = {g.db}",
        )


## Running an example

We use an input of size 2, two hidden layers of size 3 with ReLU, and an output
layer of size 2 with Identity for regression. The loss function is RSS.

For binary classification, we can use Sigmoid in the output layer, `LogLoss()`,
and binary targets with the same shape as the predictions. For regression,
we can also replace RSS with `AbsoluteError()`.

This example uses a single input vector and leaves `epocs=0`, so it performs
the forward and backward passes without training. To use the bonus features,
we can supply a matrix with one example per row or set `epocs` to a positive value.


In [12]:
relu = ReLU()
loss = RSS()
layers_spec = [LayerSpec(size, relu) for size in [3, 3]] + [LayerSpec(2, Identity())]
net = Network(2, loss, layers_spec)
show_network(net)


x = torch.tensor([1.0, 1.0])
y = torch.tensor([-1.0, 0.5])
report(net.run(x, y))


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>